In [0]:
spark.conf.set('fs.azure.account.key.janbatchsa.dfs.core.windows.net', '/8EpJrGVnWWqCWRQZJoF9XW4XGNA/OqgLybXpnsTzMD8oGHRfaYCG/CHNWAdUORPtI1hX3d9VIor+AStP4q1lw==')

In [0]:
path='abfss://raw-sales@janbatchsa.dfs.core.windows.net/sales_60_records.csv'
from pyspark.sql.types import StructType
from pyspark.sql.types import StructField
from pyspark.sql.types import StringType,IntegerType,DateType
#schema="order_id int, customer_id string,product_id string,order_date date,quantity int,price int"
schema=StructType([StructField('order_id',IntegerType(),True),\
                    StructField('customer_id',StringType(),True),\
                   StructField('product_id',StringType(),True),\
                   StructField('order_date',DateType(),True),\
                   StructField('quantity',IntegerType(),True),\
                   StructField('price',IntegerType(),True)\
                  ])

df=spark.read.csv(path,schema=schema,header=True)

display(df)

order_id,customer_id,product_id,order_date,quantity,price
1001,C002,P102,2024-01-24,4,193
1002,C003,P104,2024-01-01,1,506
1003,C005,P101,2024-01-14,4,333
1004,C002,P103,2024-01-04,2,400
1005,C005,P103,2024-01-25,1,615
1006,C002,P103,2024-01-21,4,745
1007,C003,P101,2024-01-13,2,284
1008,C003,P101,2024-01-29,5,415
1009,C002,P101,2024-01-13,2,716
1010,C002,P105,2024-01-03,4,155


In [0]:
df.createOrReplaceTempView('orders_test')

In [0]:
df.createGlobalTempView('orders_test1')

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5672376406343338>, line 1
----> 1 df.createGlobalTempView('orders_test1')

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:2115, in DataFrame.createGlobalTempView(self, name)
   2111 def createGlobalTempView(self, name: str) -> None:
   2112     command = plan.CreateView(
   2113         child=self._plan, name=name, is_global=True, replace=False
   2114     ).command(session=self._session.client)
-> 2115     _, _, ei = self._session.client.execute_command(command, self._plan.observations)
   2116     self._execution_info = ei

File /databricks/spark/python/pyspark/sql/connect/client/core.py:1589, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1587     req.user_context.user_id = self._user_id
   1588 self._set_command_in_plan(req.plan, command)
-> 1589 data, _, m

In [0]:
df.write.saveAsTable('orders_test2')

In [0]:
new_df=spark.table('orders_test2')
new_df.show()

In [0]:
writepath='abfss://silver@janbatchsa.dfs.core.windows.net/sales_data_modified'
new_df.write.format('delta').mode('overwrite').save(writepath)

In [0]:
%sql
Update orders_test2
Set customer_id='C666' Where customer_id='C002'

In [0]:
%sql
Select * from orders_test2 where customer_id='C666'

In [0]:
result=spark.sql('''
SELECT customer_id,sum(quantity)
FROM orders_test
Group By customer_id''')
result.show()

In [0]:
count=spark.sql('''select count(*) from orders_test''')
count.show()

In [0]:
%sql Update orders_test2 SET 
customer_id='C555' WHERE customer_id='C001'

In [0]:
table_info = spark.sql("DESCRIBE TABLE EXTENDED orders_test")
display(table_info)